In [3]:
pip install langdetect googletrans==4.0.0-rc1


     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ------------------------------------- 981.5/981.5 kB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 5.5 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17519 sha256=38660526799b384f2cd502a34f2d9ab0b301053273d13b07a426d61d72d1ff2a
  Stored in directory: c:\users\voquy\appdata\local\pip\cache\wheels\95\0f\04\b17a72024b56a60e499ce1a6313d283ed5ba332407155bee03
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993313 sha256=65f4e071ad7b0265e5d6faf92a907fcdaf0917a187c3b20468b1ddfee28bfaf8
  Stored in directory: c:\users\voquy\appdata\l

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.11.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.13.3 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import re
from langdetect import detect

df = pd.read_csv("data/all.csv")

# Làm sạch comment: viết thường, loại bỏ ký tự đặc biệt
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^\w\s.,!?;:()-]', '', text)
    return text

# Kiểm tra từ có phải tiếng Việt hay không
def is_vietnamese(word):
    try:
        return detect(word) == 'vi'
    except:
        return False

# Kiểm tra comment có ít nhất 1 từ tiếng Việt
def has_vietnamese(comment):
    words = comment.split()
    return any(is_vietnamese(word) for word in words)

# Kiểm tra comment có chứa ký tự tiếng Nhật không
def contains_japanese(text):
    pattern = re.compile(
        r'[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FFF\uFF66-\uFF9D]'
    )
    return bool(pattern.search(text))


cleaned_rows = []

for idx, row in df.iterrows():
    comment = row["Comment"]
    if pd.isna(comment):
        continue
    comment_clean = clean_text(comment)
    if not comment_clean.strip():
        continue
    if "https:" in comment_clean:
        continue
    if contains_japanese(comment_clean):
        continue
    if has_vietnamese(comment_clean):
        # Thay comment đã làm sạch vào DataFrame
        row["Comment"] = comment_clean
        cleaned_rows.append(row)

new_df = pd.DataFrame(cleaned_rows)

new_df.to_csv("data/cleaned/all_v1.csv", index=False, encoding="utf-8-sig")



In [7]:
import pandas as pd
import re

df = pd.read_csv("data/cleaned/vid2_v1.csv")


abbreviations = {
    "ko": "không",
    "k": "không",
    "khong": "không",
    "mk": "mình",
    "nt": "nhắn tin",
    "dc": "được",
    "đc": "được",
    "bt": "bình thường",
    "bth": "bình thường",
    "r ": "rồi ",
    "vs": "với",
    "cx": "cũng",
    "bn": "bạn",
    "mik": "mình",
    "j": "gì",
    "hok": "không",
    "kh": "không",
    "ak": "á",
    "dkh": "được không",
    "dk": "được",
    "dki": "đăng kí",
    "hk": "không",
    "thik": "thích",
    "lm": "làm",
    "s": "sao",
}

def clean_text(text):
    if pd.isnull(text):
        return ""

    
    for abbr, full in abbreviations.items():
        text = re.sub(rf"\b{abbr}\b", full, text)
    
    
    return text

df["Comment"] = df["Comment"].apply(clean_text)

df.to_csv("data/cleaned/vid2_v1.csv", index=False, encoding="utf-8-sig")
